# Canadian Portfolio Risk Analytics & Optimization Dashboard
### End-to-end risk modeling in Python — 10 years of Canadian market data

**Run each cell top-to-bottom with `Shift + Enter`.**

| Phase | Topic |
|---|---|
| 1 | Project definition & asset universe |
| 2 | Data collection & cleaning |
| 3 | Core risk metrics (return, vol, drawdown, VaR) |
| 4 | Monte Carlo simulation |
| 5 | Portfolio optimization — Efficient Frontier |
| 6 | Scenario & stress testing |
| 7 | Data export for Tableau |

> Read every markdown cell — the explanations are what you say in interviews.


---
## Phase 1 — Project Definition

### Business objective
Build a multi-asset Canadian equity portfolio risk platform that quantifies risk exposure,
simulates future portfolio paths, identifies the optimal allocation, and stress-tests
performance under market shocks.

### Key business questions
1. What is the true risk profile of this portfolio?
2. What is the worst-case loss at 95% and 99% confidence?
3. How should capital be allocated to maximize risk-adjusted return?
4. How does the portfolio behave during a market crash or rate shock?
5. Which assets drive risk vs. provide diversification?

### Target user
A retail investor, junior portfolio analyst, or risk associate at a GTA-area bank or asset manager.


In [ ]:
# Phase 1 — Asset Universe
# Document every decision in code — this is professional practice.

PROJECT = {
    "title": "Canadian Portfolio Risk Analytics Dashboard",
    "data_start": "2014-01-01",
    "benchmark": "XIC.TO",
}

ASSET_UNIVERSE = {
    # --- Canadian ETFs ---
    "XIC.TO": "iShares S&P/TSX Composite (Canadian market benchmark)",
    "XFN.TO": "iShares Financials ETF (largest TSX sector)",
    "XEG.TO": "iShares Energy ETF (commodity/energy exposure)",
    "ZAG.TO": "BMO Aggregate Bond ETF (safe-haven / low-corr asset)",
    "XIU.TO": "iShares S&P/TSX 60 (large-cap Canadian equities)",
    # --- Canadian stocks ---
    "RY.TO":  "Royal Bank of Canada (largest Canadian bank)",
    "TD.TO":  "TD Bank (major retail + US exposure)",
    "BNS.TO": "Bank of Nova Scotia (EM exposure via LatAm)",
    "CNR.TO": "Canadian National Railway (infrastructure diversifier)",
    "SU.TO":  "Suncor Energy (energy sector single-name)",
}

TICKERS = list(ASSET_UNIVERSE.keys())

print(f"Project : {PROJECT['title']}")
print(f"Assets  : {len(TICKERS)} securities")
print(f"Period  : {PROJECT['data_start']} to today")
print()
for ticker, desc in ASSET_UNIVERSE.items():
    print(f"  {ticker:<10} {desc}")


---
## Phase 2 — Data Collection & Cleaning

### Why adjusted close prices?
Adjusted close accounts for **dividends and stock splits**. Canadian bank stocks like
RY and TD pay large dividends — using raw close would significantly understate total returns.

### Cleaning pipeline (3 gates)
1. **Missing data gate** — drop any asset with >5% missing values
2. **Forward-fill** — fill short gaps from market holidays (never backward-fill = no lookahead bias)
3. **Row drop** — remove rows with any remaining NaN at start of series


In [ ]:
# Install yfinance (only needed once per Colab session)
!pip install yfinance --quiet
print("yfinance ready")


In [ ]:
# All imports for the entire notebook
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import scipy.stats as stats
import warnings
import os
from datetime import datetime

warnings.filterwarnings('ignore')

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "#f9f9f9",
    "axes.grid":        True,
    "grid.color":       "#e0e0e0",
    "grid.linewidth":   0.7,
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "font.family":      "sans-serif",
    "axes.titlesize":   13,
    "axes.labelsize":   11,
})

os.makedirs("outputs", exist_ok=True)
print("All libraries imported. Outputs folder ready.")


In [ ]:
# Download 10 years of adjusted close prices
START_DATE = '2014-01-01'
END_DATE   = datetime.today().strftime('%Y-%m-%d')

print(f'Downloading {len(TICKERS)} assets: {START_DATE} to {END_DATE}...')

raw = yf.download(
    tickers     = TICKERS,
    start       = START_DATE,
    end         = END_DATE,
    auto_adjust = True,
    progress    = True,
)

prices_raw = raw['Close'].copy()
print(f'Raw download: {prices_raw.shape[0]} days x {prices_raw.shape[1]} assets')


In [ ]:
# Clean and validate prices
def clean_prices(df, missing_threshold=0.05):
    print('--- Data Quality Report ---')

    # Gate 1: drop assets with too much missing data
    missing_pct = df.isnull().mean()
    bad = missing_pct[missing_pct > missing_threshold].index.tolist()
    if bad:
        print(f'  Dropped (>5% missing): {bad}')
        df = df.drop(columns=bad)
    else:
        print('  All assets pass missing-data gate')

    # Gate 2: forward-fill short gaps (market holidays)
    # Never backward-fill — that would be lookahead bias
    df = df.ffill()

    # Gate 3: drop rows where any price is still NaN
    rows_before = len(df)
    df = df.dropna()
    print(f'  Dropped {rows_before - len(df)} NaN rows from start of series')

    # Gate 4: sanity check — no negative prices
    if (df <= 0).any().any():
        raise ValueError('Negative or zero prices detected')
    print('  No negative prices found')

    print(f'\n  Final: {df.shape[0]} trading days x {df.shape[1]} assets')
    print(f'  Range: {df.index[0].date()} to {df.index[-1].date()}')
    print(f'  Years: {(df.index[-1] - df.index[0]).days / 365.25:.1f}')
    return df

prices = clean_prices(prices_raw)


In [ ]:
# Visualize normalized price history (base = 100)
# Normalizing lets us compare assets with very different price levels on one chart.

normalized = (prices / prices.iloc[0]) * 100

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

etfs   = ['XIC.TO', 'XFN.TO', 'XEG.TO', 'ZAG.TO', 'XIU.TO']
stocks = ['RY.TO',  'TD.TO',  'BNS.TO', 'CNR.TO', 'SU.TO']

for t in etfs:
    axes[0].plot(normalized.index, normalized[t], label=t.replace('.TO',''), lw=1.6)
axes[0].set_title('Canadian ETFs -- Normalized Price History (Base = 100)', fontweight='bold')
axes[0].set_ylabel('Indexed Price')
axes[0].legend(loc='upper left', fontsize=9)

for t in stocks:
    axes[1].plot(normalized.index, normalized[t], label=t.replace('.TO',''), lw=1.6)
axes[1].set_title('Canadian Stocks -- Normalized Price History (Base = 100)', fontweight='bold')
axes[1].set_ylabel('Indexed Price')
axes[1].set_xlabel('Date')
axes[1].legend(loc='upper left', fontsize=9)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout(pad=3)
plt.savefig('outputs/02_price_history.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved: outputs/02_price_history.png')


In [ ]:
# Correlation heatmap
# Low or negative correlation = good diversification.
# ZAG (bonds) should show low correlation with equities.

daily_returns = prices.pct_change().dropna()
corr = daily_returns.corr()
labels = [t.replace('.TO', '') for t in corr.columns]

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr.values, cmap='RdYlGn', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, label='Pearson Correlation')

ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=10)
ax.set_yticklabels(labels, fontsize=10)
ax.set_title('Asset Return Correlation Matrix', fontweight='bold', pad=14)

for i in range(len(labels)):
    for j in range(len(labels)):
        color = 'white' if abs(corr.values[i,j]) > 0.7 else 'black'
        ax.text(j, i, f'{corr.values[i,j]:.2f}', ha='center', va='center',
                fontsize=8, color=color)

plt.tight_layout()
plt.savefig('outputs/02_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/02_correlation_matrix.png')
print('Interview note: ZAG should show low/negative correlation with equities.')
print('This is why bonds are included -- they cushion equity drawdowns.')


---
## Phase 3 -- Core Risk Metrics

| Metric | Business meaning |
|---|---|
| **Annualized return** | Compounded annual growth rate |
| **Annualized volatility** | How bumpy the ride is (std dev x sqrt(252)) |
| **Maximum drawdown** | Worst peak-to-trough loss -- how bad did it get? |
| **VaR 95%** | 95% of days, loss does NOT exceed this threshold |
| **VaR 99%** | 99% of days, loss does NOT exceed this threshold |

> **Interview tip:** VaR does NOT tell you the maximum possible loss.
> It tells you the minimum loss in the worst X% of days.
>
> **Key assumption:** Historical VaR assumes the future will look like the past.
> This fails during regime changes -- which is exactly why stress testing (Phase 6) matters.


In [ ]:
# Annualized return and volatility
# Scale from daily to annual:
#   Annual return     = (1 + mean_daily)^252 - 1
#   Annual volatility = std_daily * sqrt(252)
# 252 = trading days per year (market convention)

TRADING_DAYS = 252

def annualized_return(daily_ret):
    return (1 + daily_ret.mean()) ** TRADING_DAYS - 1

def annualized_volatility(daily_ret):
    return daily_ret.std() * np.sqrt(TRADING_DAYS)

ann_returns = daily_returns.apply(annualized_return)
ann_vols    = daily_returns.apply(annualized_volatility)

rr_df = pd.DataFrame({
    'Ann. Return (%)':   (ann_returns * 100).round(2),
    'Ann. Volatility (%)': (ann_vols * 100).round(2),
    'Sharpe (rf=3%)':    ((ann_returns - 0.03) / ann_vols).round(3),
})
rr_df.index = [t.replace('.TO','') for t in rr_df.index]
rr_df = rr_df.sort_values('Sharpe (rf=3%)', ascending=False)

print('Risk-Return Summary -- Individual Assets')
print('=' * 50)
print(rr_df.to_string())
print('\nHigher Sharpe = more return per unit of risk.')
print('Risk-free rate assumption: 3% (approx. Canadian T-bill)')


In [ ]:
# Maximum Drawdown
# Drawdown = how far price has fallen from its previous peak.
# Max drawdown = the worst such fall over the full period.
# A strategy with -35% max drawdown means an investor could have
# watched their portfolio lose 35% of value before it recovered.

def max_drawdown(price_series):
    rolling_peak = price_series.cummax()
    drawdown = (price_series - rolling_peak) / rolling_peak
    return drawdown.min()

def drawdown_series(price_series):
    rolling_peak = price_series.cummax()
    return (price_series - rolling_peak) / rolling_peak

mdd = prices.apply(max_drawdown)
mdd.index = [t.replace('.TO','') for t in mdd.index]

print('Maximum Drawdown by Asset')
print('=' * 42)
for asset, val in mdd.sort_values().items():
    bar = 'x' * int(abs(val) * 40)
    print(f'  {asset:<6}  {val*100:>7.2f}%  {bar}')

print('\nInterview note: Energy assets (XEG, SU) typically show deepest drawdowns.')
print('ZAG (bonds) shows shallowest. This is the diversification benefit in action.')


In [ ]:
# Drawdown chart for the benchmark (XIC)
# Lets you see WHEN the market was in distress.
# You should be able to identify: COVID-19 crash (Mar 2020) and 2022 rate-hike selloff.

benchmark = 'XIC.TO'
dd_series = drawdown_series(prices[benchmark])

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(prices.index, prices[benchmark], color='#1f77b4', lw=1.6)
axes[0].set_title(f'{benchmark} -- Price History', fontweight='bold')
axes[0].set_ylabel('Price (CAD)')

axes[1].fill_between(dd_series.index, dd_series * 100, 0,
                     color='#d62728', alpha=0.6)
axes[1].set_title(f'{benchmark} -- Drawdown from Peak (%)', fontweight='bold')
axes[1].set_ylabel('Drawdown (%)')
axes[1].set_xlabel('Date')
axes[1].yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

worst_date = dd_series.idxmin()
worst_val  = dd_series.min() * 100
axes[1].annotate(f'  {worst_val:.1f}% ({worst_date.strftime("%b %Y")})',
                 xy=(worst_date, worst_val), fontsize=9,
                 color='#d62728', fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/03_drawdown_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/03_drawdown_benchmark.png')


In [ ]:
# Value at Risk -- Historical Method
# VaR answers: what is the minimum loss on my worst X% of days?
# Historical method: take the Nth percentile of actual past returns.
# No distributional assumptions -- most transparent VaR approach.
# Banks use this alongside parametric and Monte Carlo VaR.

def historical_var(returns, confidence):
    # confidence=0.95 -> VaR 95% -> returns the 5th percentile
    return np.percentile(returns.dropna(), (1 - confidence) * 100)

var_95 = daily_returns.apply(lambda s: historical_var(s, 0.95))
var_99 = daily_returns.apply(lambda s: historical_var(s, 0.99))

var_df = pd.DataFrame({
    'VaR 95% daily (%)': (var_95 * 100).round(3),
    'VaR 99% daily (%)': (var_99 * 100).round(3),
    'Ann. VaR 95% (%)':  (var_95 * np.sqrt(252) * 100).round(2),
})
var_df.index = [t.replace('.TO','') for t in var_df.index]
var_df = var_df.sort_values('VaR 99% daily (%)')

print('Value at Risk -- Historical Method')
print('=' * 55)
print(var_df.to_string())
print('\nExample: VaR 99% for RY means that on 99% of trading days,')
print('losses did NOT exceed that threshold.')
print('The 1% of days where losses exceeded this are tail events.')


In [ ]:
# VaR distribution chart -- makes VaR intuitive visually

asset = 'RY.TO'
ret   = daily_returns[asset].dropna() * 100
v95   = historical_var(daily_returns[asset], 0.95) * 100
v99   = historical_var(daily_returns[asset], 0.99) * 100

fig, ax = plt.subplots(figsize=(12, 5))

ax.hist(ret, bins=100, color='#4a90d9', alpha=0.7,
        edgecolor='white', linewidth=0.3, density=True)
ax.axvline(v95, color='#e67e22', lw=2, linestyle='--',
           label=f'VaR 95%: {v95:.2f}%')
ax.axvline(v99, color='#c0392b', lw=2, linestyle='--',
           label=f'VaR 99%: {v99:.2f}%')

xlim = ax.get_xlim()
x_tail = np.linspace(xlim[0], v99, 200)
kde = stats.gaussian_kde(ret)
ax.fill_between(x_tail, kde(x_tail), alpha=0.35, color='#c0392b', label='1% tail')

ax.set_title(f'{asset.replace(".TO","")} -- Daily Return Distribution with VaR',
             fontweight='bold')
ax.set_xlabel('Daily Return (%)')
ax.set_ylabel('Density')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('outputs/03_var_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'On 99% of days, {asset} loss did NOT exceed {v99:.2f}%')


In [ ]:
# Full risk summary -- save for Tableau

risk_summary = pd.DataFrame({
    'Ann. Return (%)':   (ann_returns * 100).round(2),
    'Ann. Vol (%)':      (ann_vols * 100).round(2),
    'Sharpe Ratio':      ((ann_returns - 0.03) / ann_vols).round(3),
    'Max Drawdown (%)':  (prices.apply(max_drawdown) * 100).round(2),
    'VaR 95% (%)':       (var_95 * 100).round(3),
    'VaR 99% (%)':       (var_99 * 100).round(3),
})
risk_summary.index = [t.replace('.TO','') for t in risk_summary.index]
risk_summary = risk_summary.sort_values('Sharpe Ratio', ascending=False)

print('COMPLETE RISK METRICS SUMMARY')
print('=' * 70)
print(risk_summary.to_string())

risk_summary.to_csv('outputs/phase3_risk_metrics.csv')
print('\nSaved: outputs/phase3_risk_metrics.csv')


---
## Phase 4 -- Monte Carlo Simulation

### What is Monte Carlo simulation?
Instead of predicting one future path, we simulate **1,000 possible futures** --
each one plausible given the portfolio's historical return and volatility.

### How it works
1. Calculate portfolio historical mean daily return (mu) and std dev (sigma)
2. For each simulation, draw 252 random returns from Normal(mu, sigma)
3. Compound those returns to get a portfolio value path
4. Analyze the distribution of outcomes to quantify downside risk

### Where is this used?
- **Risk management:** VaR over 10-day and 1-month horizons
- **Derivatives pricing:** Options pricing is a closed-form Monte Carlo
- **Retirement planning:** Wealth management firms use this for 30-year projections
- **Regulatory capital:** Basel III stress testing requirements

> **Interview tip:** Acknowledge the key assumption -- normally distributed returns.
> Real returns have fat tails (more extreme events than normal predicts).
> A stronger version uses a Student's t-distribution to capture this.


In [ ]:
# Define equal-weight portfolio (baseline before optimization)

n_assets = len(TICKERS)
equal_weights = np.array([1 / n_assets] * n_assets)

# Portfolio return = weighted sum of individual asset returns
portfolio_daily_ret = daily_returns[TICKERS].dot(equal_weights)

port_mean    = portfolio_daily_ret.mean()
port_std     = portfolio_daily_ret.std()
port_ann_ret = annualized_return(portfolio_daily_ret)
port_ann_vol = annualized_volatility(portfolio_daily_ret)

print('Equal-Weight Portfolio -- Baseline')
print('=' * 45)
print(f'  Assets:            {n_assets}')
print(f'  Weight per asset:  {equal_weights[0]*100:.1f}%')
print(f'  Annualized return: {port_ann_ret*100:.2f}%')
print(f'  Ann. volatility:   {port_ann_vol*100:.2f}%')
print(f'  Sharpe (rf=3%):    {(port_ann_ret - 0.03)/port_ann_vol:.3f}')


In [ ]:
# Run Monte Carlo simulation
# 1,000 paths x 252 trading days, starting value $100,000

np.random.seed(42)  # Set seed for reproducibility

N_SIMULATIONS  = 1000
N_DAYS         = 252
INITIAL_VALUE  = 100_000

# Pre-allocate results matrix: rows=days, columns=simulations
sim_matrix = np.zeros((N_DAYS + 1, N_SIMULATIONS))
sim_matrix[0] = INITIAL_VALUE

print(f'Running {N_SIMULATIONS} simulations x {N_DAYS} trading days...')

for sim in range(N_SIMULATIONS):
    # Draw N_DAYS random daily returns from normal distribution
    rand_returns = np.random.normal(loc=port_mean, scale=port_std, size=N_DAYS)
    # cumprod() compounds daily returns into a value path
    sim_matrix[1:, sim] = INITIAL_VALUE * np.cumprod(1 + rand_returns)

final_values = sim_matrix[-1]
print(f'Done. Final value range: ${final_values.min():,.0f} to ${final_values.max():,.0f}')
print(f'Median final value: ${np.median(final_values):,.0f}')


In [ ]:
# Visualize Monte Carlo results

p5  = np.percentile(final_values, 5)
p50 = np.percentile(final_values, 50)
p95 = np.percentile(final_values, 95)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Left: simulation paths ---
ax = axes[0]
sample = np.random.choice(N_SIMULATIONS, 200, replace=False)
for i in sample:
    ax.plot(sim_matrix[:, i], color='#4a90d9', alpha=0.04, lw=0.8)

ax.plot(np.percentile(sim_matrix, 5,  axis=1), color='#c0392b', lw=2.2,
        linestyle='--', label=f'5th pct  ${p5:,.0f}')
ax.plot(np.percentile(sim_matrix, 50, axis=1), color='#27ae60', lw=2.5,
        label=f'Median   ${p50:,.0f}')
ax.plot(np.percentile(sim_matrix, 95, axis=1), color='#2980b9', lw=2.2,
        linestyle='--', label=f'95th pct ${p95:,.0f}')
ax.axhline(INITIAL_VALUE, color='gray', lw=1.2, linestyle=':', alpha=0.7)
ax.set_title(f'Monte Carlo: {N_SIMULATIONS} Portfolio Paths (1 Year)', fontweight='bold')
ax.set_xlabel('Trading Days')
ax.set_ylabel('Portfolio Value (CAD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1000:.0f}K'))
ax.legend(fontsize=9)

# --- Right: distribution of final values ---
ax2 = axes[1]
ax2.hist(final_values, bins=60, color='#4a90d9', alpha=0.75,
         edgecolor='white', linewidth=0.3)
ax2.axvline(p5,  color='#c0392b', lw=2,   linestyle='--', label=f'5th pct  ${p5:,.0f}')
ax2.axvline(p50, color='#27ae60', lw=2.5,                  label=f'Median   ${p50:,.0f}')
ax2.axvline(p95, color='#2980b9', lw=2,   linestyle='--', label=f'95th pct ${p95:,.0f}')
ax2.axvline(INITIAL_VALUE, color='gray',  lw=1.5, linestyle=':', label='Initial $100K')
ax2.set_title('Distribution of Final Portfolio Values (Day 252)', fontweight='bold')
ax2.set_xlabel('Portfolio Value (CAD)')
ax2.set_ylabel('Frequency')
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1000:.0f}K'))
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('outputs/04_monte_carlo.png', dpi=150, bbox_inches='tight')
plt.show()

prob_loss = (final_values < INITIAL_VALUE).mean() * 100
print(f'Probability of loss after 1 year:  {prob_loss:.1f}%')
print(f'Expected value (median):           ${p50:,.0f}')
print(f'Worst 5% scenario ends at:         ${p5:,.0f}  ({(p5/INITIAL_VALUE-1)*100:.1f}%)')
print(f'Best 5% scenario ends at:          ${p95:,.0f}  ({(p95/INITIAL_VALUE-1)*100:.1f}%)')


In [ ]:
# Save Monte Carlo percentiles for Tableau
mc_percentiles = pd.DataFrame({
    'Day': range(N_DAYS + 1),
    'P5':  np.percentile(sim_matrix, 5,  axis=1),
    'P25': np.percentile(sim_matrix, 25, axis=1),
    'P50': np.percentile(sim_matrix, 50, axis=1),
    'P75': np.percentile(sim_matrix, 75, axis=1),
    'P95': np.percentile(sim_matrix, 95, axis=1),
    'Initial': INITIAL_VALUE,
})
mc_percentiles.to_csv('outputs/phase4_monte_carlo_percentiles.csv', index=False)
print('Saved: outputs/phase4_monte_carlo_percentiles.csv')


---
## Phase 5 -- Portfolio Optimization (Modern Portfolio Theory)

### What is MPT?
Harry Markowitz (Nobel Prize 1990): for any target return, there exists a portfolio
with the minimum possible risk. The **Efficient Frontier** is the set of all such portfolios.

### The Sharpe Ratio
```
Sharpe = (Portfolio Return - Risk-Free Rate) / Portfolio Volatility
```
The **maximum Sharpe portfolio** (tangency portfolio) is the rational investor's optimal choice.

### Two inputs, that is all MPT needs
1. **Expected returns** -- annualized mean return per asset
2. **Covariance matrix** -- how assets move relative to each other

> **Interview relevance:** MPT is foundational for portfolio construction, risk budgeting,
> and capital allocation roles at asset managers, banks, and insurance companies.


In [ ]:
# Generate 10,000 random portfolio weight combinations
# This maps out the feasible portfolio space.

N_PORTFOLIOS = 10_000
RF_RATE      = 0.03

np.random.seed(0)

# The two inputs MPT needs
mean_returns = daily_returns[TICKERS].mean() * TRADING_DAYS   # annualized
cov_matrix   = daily_returns[TICKERS].cov()  * TRADING_DAYS   # annualized

results      = np.zeros((N_PORTFOLIOS, 3))           # [return, vol, sharpe]
weight_store = np.zeros((N_PORTFOLIOS, len(TICKERS)))

print(f'Simulating {N_PORTFOLIOS:,} random portfolios...')

for i in range(N_PORTFOLIOS):
    w = np.random.random(len(TICKERS))
    w /= w.sum()                                   # normalize to sum to 1

    port_ret    = np.dot(w, mean_returns)
    port_vol    = np.sqrt(w.T @ cov_matrix.values @ w)
    port_sharpe = (port_ret - RF_RATE) / port_vol

    results[i]       = [port_ret, port_vol, port_sharpe]
    weight_store[i]  = w

print('Done.')
print(f'Return range:  {results[:,0].min()*100:.1f}% to {results[:,0].max()*100:.1f}%')
print(f'Vol range:     {results[:,1].min()*100:.1f}% to {results[:,1].max()*100:.1f}%')
print(f'Sharpe range:  {results[:,2].min():.3f} to {results[:,2].max():.3f}')


In [ ]:
# Identify the optimal portfolios

max_sharpe_idx     = results[:, 2].argmax()
max_sharpe_ret     = results[max_sharpe_idx, 0]
max_sharpe_vol     = results[max_sharpe_idx, 1]
max_sharpe_weights = weight_store[max_sharpe_idx]

min_vol_idx     = results[:, 1].argmin()
min_vol_ret     = results[min_vol_idx, 0]
min_vol_vol     = results[min_vol_idx, 1]
min_vol_weights = weight_store[min_vol_idx]

print('OPTIMAL PORTFOLIOS')
print('=' * 55)
print(f'\nMax Sharpe Ratio Portfolio')
print(f'  Return:     {max_sharpe_ret*100:.2f}%')
print(f'  Volatility: {max_sharpe_vol*100:.2f}%')
print(f'  Sharpe:     {results[max_sharpe_idx,2]:.3f}')
print(f'  Weights:')
for ticker, w in zip(TICKERS, max_sharpe_weights):
    if w > 0.005:
        print(f'    {ticker:<10} {w*100:.1f}%')

print(f'\nMin Volatility Portfolio')
print(f'  Return:     {min_vol_ret*100:.2f}%')
print(f'  Volatility: {min_vol_vol*100:.2f}%')
print(f'  Sharpe:     {results[min_vol_idx,2]:.3f}')
print(f'  Weights:')
for ticker, w in zip(TICKERS, min_vol_weights):
    if w > 0.005:
        print(f'    {ticker:<10} {w*100:.1f}%')


In [ ]:
# Efficient Frontier chart

fig, ax = plt.subplots(figsize=(13, 8))

sc = ax.scatter(results[:,1]*100, results[:,0]*100,
                c=results[:,2], cmap='viridis', alpha=0.35, s=8, linewidths=0)
plt.colorbar(sc, ax=ax, label='Sharpe Ratio')

# Individual assets
for i, ticker in enumerate(TICKERS):
    ax.scatter(ann_vols[ticker]*100, ann_returns[ticker]*100,
               s=90, zorder=5, marker='D', color='#555')
    ax.annotate(ticker.replace('.TO',''),
                xy=(ann_vols[ticker]*100, ann_returns[ticker]*100),
                xytext=(5, 3), textcoords='offset points', fontsize=8)

ax.scatter(max_sharpe_vol*100, max_sharpe_ret*100,
           s=250, marker='*', color='#e74c3c', zorder=10,
           label=f'Max Sharpe ({results[max_sharpe_idx,2]:.2f})', edgecolors='white')
ax.scatter(min_vol_vol*100, min_vol_ret*100,
           s=200, marker='^', color='#2980b9', zorder=10,
           label=f'Min Volatility ({results[min_vol_idx,2]:.2f})', edgecolors='white')

ax.set_title('Efficient Frontier -- Canadian Portfolio Universe', fontweight='bold')
ax.set_xlabel('Annualized Volatility (%)')
ax.set_ylabel('Annualized Return (%)')
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('outputs/05_efficient_frontier.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/05_efficient_frontier.png')
print('Red star = max-Sharpe portfolio (most return per unit of risk).')
print('Blue triangle = min-volatility portfolio (safest allocation).')


In [ ]:
# Save optimization results for Tableau

pd.DataFrame(results, columns=['Return','Volatility','Sharpe']).to_csv(
    'outputs/phase5_all_portfolios.csv', index=False)

optimal_weights_df = pd.DataFrame({
    'Ticker':      TICKERS,
    'Asset':       [ASSET_UNIVERSE[t] for t in TICKERS],
    'MaxSharpe_W': max_sharpe_weights,
    'MinVol_W':    min_vol_weights,
    'EqualW':      equal_weights,
})
optimal_weights_df.to_csv('outputs/phase5_optimal_weights.csv', index=False)
print('Saved: phase5_all_portfolios.csv and phase5_optimal_weights.csv')
print()
print(optimal_weights_df.to_string(index=False))


---
## Phase 6 -- Scenario & Stress Testing

VaR and volatility describe normal conditions. Stress testing asks:
**what happens when conditions are NOT normal?**

This is required by Canadian regulators (OSFI) and is a core part of risk management
at every Canadian bank and insurance company.

| Scenario | Description | Real-world analogue |
|---|---|---|
| **Market crash** | Sudden -20% shock to all equity positions | COVID-19 (Mar 2020), GFC (2008) |
| **High volatility** | Double historical volatility, same mean | VIX spike events |
| **Rate shock** | Bonds -10%, equities -5% | BoC rapid rate hikes (2022) |

> **Interview tip:** Mention OSFI's requirement for annual stress tests on capital ratios.
> Shows you understand the regulatory context -- a big signal for risk analyst roles at banks.


In [ ]:
# Apply stress scenarios to the Max Sharpe portfolio

portfolio_weights = max_sharpe_weights

SCENARIOS = {
    'Baseline': {'shock': None},
    'Market Crash (-20%)': {
        'shock': {t: -0.20 for t in TICKERS if t != 'ZAG.TO'}
    },
    'High Volatility (2x)': {'shock': 'double_vol'},
    'Rate Shock (BoC)': {
        'shock': {'ZAG.TO': -0.10,
                  **{t: -0.05 for t in TICKERS if t != 'ZAG.TO'}}
    },
}

scenario_results = {}

for name, config in SCENARIOS.items():
    shock = config['shock']

    if shock is None:
        stressed = daily_returns[TICKERS].copy()

    elif shock == 'double_vol':
        means = daily_returns[TICKERS].mean()
        stressed = (daily_returns[TICKERS] - means) * 2 + means

    else:
        stressed = daily_returns[TICKERS].copy()
        shock_row = pd.DataFrame(
            [{t: shock.get(t, 0) for t in TICKERS}], columns=TICKERS
        )
        stressed = pd.concat([stressed, shock_row], ignore_index=True)

    port_ret = stressed.dot(portfolio_weights)

    scenario_results[name] = {
        'Ann. Return (%)':   annualized_return(port_ret) * 100,
        'Ann. Vol (%)':      annualized_volatility(port_ret) * 100,
        'Sharpe Ratio':      (annualized_return(port_ret) - 0.03) / annualized_volatility(port_ret),
        'VaR 95% (%)':       historical_var(port_ret, 0.95) * 100,
        'VaR 99% (%)':       historical_var(port_ret, 0.99) * 100,
        'Max Drawdown (%)':  max_drawdown(pd.Series((1 + port_ret).cumprod())) * 100,
    }

scenario_df = pd.DataFrame(scenario_results).T.round(3)
print('STRESS TEST RESULTS -- MAX SHARPE PORTFOLIO')
print('=' * 70)
print(scenario_df.to_string())
scenario_df.to_csv('outputs/phase6_scenario_results.csv')
print('\nSaved: outputs/phase6_scenario_results.csv')


In [ ]:
# Stress test visualization

fig, axes = plt.subplots(1, 3, figsize=(15, 6))
scenarios = list(scenario_results.keys())
colors    = ['#2ecc71', '#e74c3c', '#e67e22', '#8e44ad']
metrics   = ['Ann. Return (%)', 'Ann. Vol (%)', 'VaR 99% (%)']
titles    = ['Annualized Return', 'Annualized Volatility', 'VaR 99% Daily']

for col, (metric, title) in enumerate(zip(metrics, titles)):
    vals = [scenario_results[s][metric] for s in scenarios]
    bars = axes[col].bar(range(len(scenarios)), vals, color=colors,
                         edgecolor='white', linewidth=0.5)
    axes[col].set_title(title, fontweight='bold')
    axes[col].set_xticks(range(len(scenarios)))
    axes[col].set_xticklabels([s.split('(')[0].strip() for s in scenarios],
                              rotation=15, ha='right', fontsize=9)
    axes[col].set_ylabel(metric)
    for bar, val in zip(bars, vals):
        axes[col].text(bar.get_x() + bar.get_width()/2,
                       bar.get_height() + 0.05,
                       f'{val:.2f}%', ha='center', va='bottom',
                       fontsize=8, fontweight='bold')

plt.suptitle('Stress Test Results -- Max Sharpe Portfolio',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('outputs/06_stress_test.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/06_stress_test.png')


---
## Phase 7 -- Export All Data for Tableau

| File | Tableau use |
|---|---|
| `tableau_prices.csv` | Price history line chart |
| `tableau_returns.csv` | Return distribution, rolling metrics |
| `tableau_risk_metrics.csv` | Asset-level risk/return scatter |
| `tableau_optimal_weights.csv` | Portfolio allocation pie chart |
| `tableau_monte_carlo.csv` | Fan chart of simulated paths |
| `tableau_scenarios.csv` | Scenario comparison bar chart |
| `tableau_drawdowns.csv` | Drawdown area chart |

All files use **long format** (one row per observation) -- Tableau prefers this
over wide format because it maps directly to dimensions and measures.


In [ ]:
# Export all Tableau-ready CSVs

# 1. Prices (long format)
prices_long = prices.reset_index().melt(
    id_vars='Date', var_name='Ticker', value_name='Adj_Close')
prices_long['Asset_Name'] = prices_long['Ticker'].map(ASSET_UNIVERSE)
prices_long.to_csv('outputs/tableau_prices.csv', index=False)
print(f'tableau_prices.csv          {len(prices_long):,} rows')

# 2. Daily returns (long format)
returns_long = daily_returns[TICKERS].reset_index().melt(
    id_vars='Date', var_name='Ticker', value_name='Daily_Return')
returns_long.to_csv('outputs/tableau_returns.csv', index=False)
print(f'tableau_returns.csv         {len(returns_long):,} rows')

# 3. Risk metrics
risk_export = risk_summary.reset_index()
risk_export.columns = ['Ticker'] + list(risk_summary.columns)
risk_export.to_csv('outputs/tableau_risk_metrics.csv', index=False)
print(f'tableau_risk_metrics.csv    {len(risk_export):,} rows')

# 4. Optimal weights
optimal_weights_df.to_csv('outputs/tableau_optimal_weights.csv', index=False)
print(f'tableau_optimal_weights.csv {len(optimal_weights_df):,} rows')

# 5. Monte Carlo percentiles (already saved)
print(f'phase4_monte_carlo_percentiles.csv  (already saved)')

# 6. Scenario results
scenario_export = scenario_df.reset_index()
scenario_export.columns = ['Scenario'] + list(scenario_df.columns)
scenario_export.to_csv('outputs/tableau_scenarios.csv', index=False)
print(f'tableau_scenarios.csv       {len(scenario_export):,} rows')

# 7. Drawdowns (long format)
dd_all  = pd.DataFrame({t: drawdown_series(prices[t]) for t in TICKERS})
dd_long = dd_all.reset_index().melt(
    id_vars='Date', var_name='Ticker', value_name='Drawdown')
dd_long.to_csv('outputs/tableau_drawdowns.csv', index=False)
print(f'tableau_drawdowns.csv       {len(dd_long):,} rows')

print('\n' + '='*50)
print('ALL TABLEAU EXPORTS COMPLETE')
print('='*50)
import os
for f in sorted(os.listdir('outputs')):
    size = os.path.getsize(f'outputs/{f}')
    print(f'  {f:<45} {size/1024:.1f} KB')


In [ ]:
# Download all files as a ZIP from Colab

import shutil
shutil.make_archive('portfolio_risk_exports', 'zip', 'outputs')
print('portfolio_risk_exports.zip created')

try:
    from google.colab import files
    files.download('portfolio_risk_exports.zip')
    print('Download triggered -- check your browser downloads folder')
except ImportError:
    print('Not in Colab -- file is at: portfolio_risk_exports.zip')


---
## Project Complete

### What you built
| Phase | Deliverable |
|---|---|
| 2 | 10-year cleaned price dataset, correlation matrix |
| 3 | Full risk metrics: return, vol, Sharpe, drawdown, VaR 95/99 |
| 4 | Monte Carlo simulation: 1,000 paths, percentile fan chart |
| 5 | Efficient frontier, max-Sharpe and min-vol portfolios |
| 6 | 3-scenario stress test with full metric impact |
| 7 | 7 Tableau-ready CSV exports |

---
### Resume bullet points

- Built an end-to-end portfolio risk analytics platform in Python analyzing 10 Canadian equities and ETFs over 10 years, computing VaR (95/99%), maximum drawdown, and Sharpe-optimized allocations using Modern Portfolio Theory
- Ran 1,000-path Monte Carlo simulation to quantify 1-year downside risk and developed 3-scenario stress tests (market crash, high volatility, rate shock), exposing key tail risks in Canadian equity exposure
- Exported 7 structured datasets to Tableau to power an interactive risk dashboard for portfolio managers and risk analysts

---
### Interview answer: Walk me through your project

*I built a risk analytics platform for a 10-asset Canadian portfolio. I started by collecting 10 years of adjusted price data using yfinance, then calculated the core risk metrics a portfolio manager cares about: annualized return, volatility, drawdown, and VaR at 95% and 99% confidence. I then ran a Monte Carlo simulation to project 1,000 possible portfolio paths one year forward, and used Modern Portfolio Theory to find the Sharpe-optimal allocation. Finally, I stress-tested the portfolio under three scenarios -- a market crash, a high-volatility environment, and a Bank of Canada rate shock -- to quantify tail risk. Everything feeds into a Tableau dashboard.*

### Interview answer: What would you improve?

*Three things. First, I would use a Student t-distribution in Monte Carlo instead of normal -- it better captures fat tails in real return data. Second, I would add factor decomposition to break down returns into market, sector, and idiosyncratic components. Third, I would integrate a real-time data feed so the dashboard updates automatically, which is how it would work in a production environment at a bank.*
